# Node 01 · 权威微观态指认 (`master_microstates.csv`)

**目的**:对 Node 00 的 1,606 个身份,用 Dimorphite-DL 在 pH 7.4 下**拍板每个分子一个权威微观态**(含显式净电荷),并白纸黑字记录。这是整个重建的核心科学决定——电荷在此确定,下游不得再猜。

**已确认政策**:后端 Dimorphite-DL;pH 7.4;保留单一主导微观态(多态则标记复核,不悄悄选);仅质子化(互变异构/未定义立体 → 标记复核,不自动处理)。

**不变量(fail-closed)**:质子化可增减 H、改电荷,但不得改重原子骨架(heavy-atom formula + HetAtomProtomer hash 核验)。

**附带**:凭权威电荷,把 Node 00 里"待定"的 `qm_status` 升级为**权威判定** `qm_final_status`。


## 0 · 全局配置与安全开关

In [1]:
# ── 全局配置与安全开关 ──────────────────────────────────
from pathlib import Path
import os, sys, json, hashlib, platform, datetime
# 安全默认：不做任何写/构建/运行，只读检查
DRY_RUN = False
# 根目录：默认取 notebook 所在目录；HPC 上应为 /mnt/hpc/home/wanghongmei/lang2
PROJECT_ROOT = Path(os.environ.get("PRISM_ROOT", "..")).resolve()
CORE_DIR    = PROJECT_ROOT / "core"
RESULTS_DIR = PROJECT_ROOT / "results"
STAGE_DIRS = [
    "00_scope", "01_microstates", "02_master_molecules", "03_pdbqt",
    "04_docking", "05_broad_spectrum_ranking", "06_md_selection",
    "07_parameterization", "08_md", "09_qc",
]
def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# ── 本步(Node 01)专用路径与参数 ──────────────────────────
IN_CSV       = RESULTS_DIR / "00_scope" / "all_candidates_v3_start.csv"   # 输入=Node00 产物
MICRO_DIR    = RESULTS_DIR / "01_microstates"
OUT_CSV      = MICRO_DIR / "master_microstates.csv"
OUT_MANIFEST = MICRO_DIR / "master_microstates.manifest.json"
OUT_TRACE    = MICRO_DIR / "master_microstates.trace.json"
PH           = 7.4
PKA_PRECISION= 1.0
PER_MOL_TIMEOUT_S = 20   # 单分子超时(秒)：多氮杂环组合爆炸时不拖垮整批
LOG_EVERY    = 200       # 每处理 N 个打印一次进度
DOMINANT_RULE = True     # 用生理 pH7.4 电荷规则在候选态中选主导态(纠正词典序占位的过度去质子)

print(f"DRY_RUN       = {DRY_RUN}")
print(f"PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"IN_CSV        = {IN_CSV}")
print(f"OUT_CSV       = {OUT_CSV}")
print(f"pH / precision= {PH} / {PKA_PRECISION}")
print(f"时间(UTC)     = {utc_now()}")

DRY_RUN       = False
PROJECT_ROOT  = /mnt/hpc/home/wanghongmei/lang2
IN_CSV        = /mnt/hpc/home/wanghongmei/lang2/results/00_scope/all_candidates_v3_start.csv
OUT_CSV       = /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/master_microstates.csv
pH / precision= 7.4 / 1.0
时间(UTC)     = 2026-07-24T04:10:07Z


## 1 · 环境 & provenance 快照(留痕)

In [3]:
import pandas as pd
try:
    import rdkit
    from rdkit import Chem
    from rdkit.Chem import rdMolHash
    RDKIT_VER = rdkit.__version__
except Exception as e:
    Chem = None; rdMolHash = None; RDKIT_VER = f"UNAVAILABLE ({e})"
try:
    import dimorphite_dl
    DIMORPHITE_VER = getattr(dimorphite_dl, "__version__", "installed")
except Exception as e:
    DIMORPHITE_VER = f"UNAVAILABLE ({e})"

env = {
    "timestamp_utc": utc_now(),
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "rdkit": RDKIT_VER,
    "dimorphite_dl": DIMORPHITE_VER,
    "platform": platform.platform(),
}
print(json.dumps(env, indent=2, ensure_ascii=False))
assert Chem is not None, "需要 RDKit"
assert "UNAVAILABLE" not in str(DIMORPHITE_VER), "需要 Dimorphite-DL(pip install dimorphite-dl)"

{
  "timestamp_utc": "2026-07-24T04:10:28Z",
  "python": "3.10.19",
  "pandas": "2.3.3",
  "rdkit": "2025.09.4",
  "dimorphite_dl": "2.0.2",
  "platform": "Linux-3.10.0-862.el7.x86_64-x86_64-with-glibc2.17"
}


## 2 · 只读读取输入 & 记录哈希(留痕)

In [4]:
assert IN_CSV.exists(), f"找不到输入:{IN_CSV}(先跑 Node 00)"
in_hash = file_sha256(IN_CSV)
in_df   = pd.read_csv(IN_CSV)
print("输入文件      :", IN_CSV)
print("输入 sha256   :", in_hash)
print("输入形状      :", in_df.shape)
assert "canonical_smiles" in in_df.columns, "输入缺 canonical_smiles"
print("出身分布      :", in_df["is_bbs_guided"].value_counts().to_dict()
      if "is_bbs_guided" in in_df.columns else "n/a")
print("Node00 qm_status:", in_df["qm_status"].value_counts(dropna=False).to_dict()
      if "qm_status" in in_df.columns else "n/a")

输入文件      : /mnt/hpc/home/wanghongmei/lang2/results/00_scope/all_candidates_v3_start.csv
输入 sha256   : 64c1d88ad3986ef4de9a775af9a7c0b24b965170bc3ec643469d3045f2196e78
输入形状      : (1606, 32)
出身分布      : {1: 932, 0: 674}
Node00 qm_status: {'none': 1245, 'ok': 337, 'needs_recompute': 24}


## 3 · 导入 core 算法(不内联)

In [5]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from core import Node01_microstate_enumerate as n1
print("Node01 version :", n1.NODE01_VERSION)
print("backend        : Dimorphite-DL (core.default_protonate)")
print("policy         : pH", PH, "| single dominant | protonation-only | flag tautomer/stereo")

# ── 后端自检:批量前先用已知分子探一下,尽早暴露 API/版本问题 ──────────
#    乙酸 CC(=O)O 在 pH 7.4 应去质子为 CC(=O)[O-]
try:
    probe = n1.selftest_backend(n1.default_protonate, ph=PH, precision=PKA_PRECISION)
    print("backend self-test:", probe)
    assert probe["deprotonated_as_expected"], \
        "Dimorphite 可导入但行为异常(乙酸未去质子)——请核对版本/参数。"
    print("[OK] 后端自检通过,可进入批量")
except Exception as e:
    import dimorphite_dl
    print("[后端自检失败] ", repr(e))
    print("dimorphite_dl 暴露的属性:",
          [a for a in dir(dimorphite_dl) if not a.startswith("_")])
    print("→ 把上面这行属性发给我,我据此把 core.default_protonate 锁到你这个版本的确切 API。")
    raise

Node01 version : 0.9.2
backend        : Dimorphite-DL (core.default_protonate)
policy         : pH 7.4 | single dominant | protonation-only | flag tautomer/stereo
backend self-test: {'probe': 'CC(=O)O', 'states': ['CC(=O)[O-]'], 'deprotonated_as_expected': True}
[OK] 后端自检通过,可进入批量


## 4 · 指认微观态(核心一步)

`n1.run` 内部:逐分子 Dimorphite 质子化 → 只保留骨架与输入一致的态 → 单态=assigned/unchanged、多态=ambiguous(标记复核)→ 记录净电荷、未定义立体 → 派生 `qm_final_status` → QC 门 → 写 manifest+trace。
`DRY_RUN=True` 时**不落盘**,仅返回结果供核验(1,606 个分子,Dimorphite 很快,通常一两分钟内)。

In [6]:
out, manifest = n1.run(
    IN_CSV, OUT_CSV, OUT_MANIFEST, OUT_TRACE,
    pd, Chem,
    protonate_fn=n1.default_protonate,   # 如你的 Dimorphite API 不同,只改 core 里这一个函数
    rdMolHash=rdMolHash,
    ph=PH, precision=PKA_PRECISION,
    timeout_s=PER_MOL_TIMEOUT_S,         # 单分子超时;进度会实时打印
    log_every=LOG_EVERY,
    dominant_rule=DOMINANT_RULE,   # v0.4.0:主导电荷规则
    dry_run=DRY_RUN,
)
s = manifest["stats"]
print("总数              :", s["n_total"])
print("protonation_status:", s["protonation_status_counts"])
print("真正指认成功      :", s["n_microstates_assigned"])
print("失败(fallback)   :", s["n_failed"], " 其中超时:", s["n_timeouts"])
print("被拒非法态总数    :", s["total_invalid_states_rejected"])
print("净电荷分布        :", s["net_charge_distribution"])
print("需复核            :", s["n_needs_review"], " 未定义立体:", s["n_stereo_undefined"])
print("resolved_by       :", s.get("resolved_by_counts", {}))
print("qm_final_status   :", s["qm_final_status_counts"])
print("QC 通过           :", manifest["qc_gate"]["passed"], manifest["qc_gate"]["problems"])

  [200/1606] elapsed=   22s  assigned=200  timeouts=0
  [400/1606] elapsed=   35s  assigned=400  timeouts=0
  [600/1606] elapsed=   45s  assigned=600  timeouts=0
  [800/1606] elapsed=   55s  assigned=800  timeouts=0
  [1000/1606] elapsed=   65s  assigned=1000  timeouts=0
  [1200/1606] elapsed=   75s  assigned=1200  timeouts=0
  [1400/1606] elapsed=   89s  assigned=1400  timeouts=0
  [1600/1606] elapsed=   96s  assigned=1600  timeouts=0
  [1606/1606] elapsed=   96s  assigned=1606  timeouts=0
总数              : 1606
protonation_status: {'unchanged': 1112, 'assigned': 494}
真正指认成功      : 1606
失败(fallback)   : 0  其中超时: 0
被拒非法态总数    : 6
净电荷分布        : {'0': 1118, '1': 284, '2': 114, '-1': 60, '3': 17, '-2': 11, '4': 2}
需复核            : 117  未定义立体: 102
resolved_by       : {'rule': 1398, 'rule_constructed_site': 173, 'rule_input': 23, 'rule_constructed': 9, 'rule_positional': 3}
qm_final_status   : {'none': 1245, 'valid': 235, 'recompute': 126}
QC 通过           : True []


## 5 · 留痕核验:不变量 / 无污染 / 复核清单

In [7]:
# 5a 不变量:凡 assigned/unchanged,skeleton_ok 必须为 True
bad = out[(out["protonation_status"].isin(["assigned","unchanged"])) & (out["skeleton_ok"] != True)]
assert len(bad) == 0, f"骨架不变量被破坏:{len(bad)} 行"
print("[OK] 骨架不变量:所有 assigned/unchanged 行均保持重原子骨架")

# 5b 防污染:输出中不得出现任何 docking 列
DOCK = ["Broad_Spectrum_Score","E_SARS_CoV_2","E_SARS_CoV_1","E_MERS_CoV","Docking_Rank","Is_Final_Top"]
leak = [c for c in DOCK if c in out.columns]
assert not leak, f"docking 列泄漏:{leak}"
print("[OK] 无 docking 列泄漏")

# 5c QM 权威判定增量:Node00=ok 但权威电荷非零 → 需重算
if "qm_status" in out.columns and "qm_final_status" in out.columns:
    flipped = out[(out["qm_status"]=="ok") & (out["qm_final_status"]=="recompute")]
    print(f"[信息] Node00=ok 但权威电荷非零 → 需重算 QM 的分子数:{len(flipped)}")

# 5d 失败/超时清单(多氮杂环组合爆炸)——可调大 PER_MOL_TIMEOUT_S 单独重试
failed = out[out["protonation_status"].isin(["failed_protonation","failed_parse"])]
n_to = int(out["review_reason"].fillna("").str.contains("timed out").sum())
print(f"\n失败 {len(failed)} 个(其中超时 {n_to} 个):")
fcols = [c for c in ["name","canonical_smiles","protonation_status","n_states_rejected","review_reason"] if c in out.columns]
if len(failed):
    print(failed[fcols].head(10).to_string(index=False))

# 5e 复核清单预览(交人工看的)
rev = out[out["needs_review"]]
print(f"\n需人工复核 {len(rev)} 个,原因分布:")
print(rev["review_reason"].value_counts().head(10).to_string())
rcols = [c for c in ["name","canonical_smiles","microstate_smiles","net_charge","protonation_status","stereo_undefined_centers","review_reason"] if c in out.columns]
rev[rcols].head(10)

[OK] 骨架不变量:所有 assigned/unchanged 行均保持重原子骨架
[OK] 无 docking 列泄漏
[信息] Node00=ok 但权威电荷非零 → 需重算 QM 的分子数:102

失败 0 个(其中超时 0 个):

需人工复核 117 个,原因分布:
review_reason
1 undefined stereocentre(s)                                                                                                       31
2 undefined stereocentre(s)                                                                                                       20
1 undefined stereocentre(s); 1 phantom centre(s) excluded (protonated N=1, bridgehead=0)                                          19
3 undefined stereocentre(s)                                                                                                        9
2 undefined stereocentre(s); 1 phantom centre(s) excluded (protonated N=1, bridgehead=0)                                           8
|net_charge|=3 >= 3 (charge suspect)                                                                                               8
4 undefined stereocentre(s); 1 phantom centre(s

,name,canonical_smiles,microstate_smiles,net_charge,protonation_status,stereo_undefined_centers,review_reason
105,pool_mol_106,O=C1CC(NC(=O)c2cc(Br)cc([N+](=O)[O-])c2N[C@@H]...,O=C1CC(NC(=O)c2cc(Br)cc([N+](=O)[O-])c2N[C@@H]...,0,unchanged,2,2 undefined stereocentre(s)
109,pool_mol_110,NC1CN(c2cncc(-c3cncc(N4CC(N)CC(F)(F)C4)c3)c2)C...,[NH3+]C1CN(c2cncc(-c3cncc(N4CC([NH3+])CC(F)(F)...,2,assigned,2,2 undefined stereocentre(s)
116,pool_mol_117,CNc1cncc(C(=O)N2C[C@@H](Nc3c(C(=O)N4CC5C(C4)C5...,CNc1cncc(C(=O)N2C[C@@H](Nc3c(C(=O)N4CC5C(C4)C5...,0,unchanged,2,2 undefined stereocentre(s)
169,pool_mol_170,C[C@@H](C(=O)OC(=O)[C@H](C)N1CCC2(CCNCC2)CC1)N...,C[C@@H](C(=O)OC(=O)[C@H](C)[NH+]1CCC2(CC[NH2+]...,4,assigned,0,"1 phantom centre(s) excluded (protonated N=1, ..."
185,pool_mol_186,CC(C)C(C(=O)N[C@@H]1CCCN([N+](=O)[O-])C1)C(=O)...,CC(C)C(C(=O)N[C@@H]1CCCN([N+](=O)[O-])C1)C(=O)...,2,assigned,1,1 undefined stereocentre(s); 1 phantom centre(...
234,pool_mol_235,C[C@H](C(=O)[C@H](C)N1CCC2(CCNCC2)CC1)C(=O)N1C...,C[C@H](C(=O)[C@H](C)[NH+]1CCC2(CC[NH2+]CC2)CC1...,3,assigned,0,|net_charge|=3 >= 3 (charge suspect)
261,pool_mol_262,CNc1cncc(C(=O)N2C[C@H](Nc3c(C(=O)N4CC5C(C4)C5(...,CNc1cncc(C(=O)N2C[C@H](Nc3c(C(=O)N4CC5C(C4)C5(...,0,unchanged,2,2 undefined stereocentre(s)
283,pool_mol_284,O=C1CC(C(=O)C(=O)[C@H]2CCCN(C(=O)C3CCNC(=O)C3)...,O=C1CC(C(=O)C(=O)[C@H]2CCCN(C(=O)C3CCNC(=O)C3)...,0,unchanged,2,2 undefined stereocentre(s)
290,pool_mol_291,O=c1[nH]c2cc(C3CNC3)c(C3CNC3)cc2c(=O)n1C1CNC1,O=c1[nH]c2cc(C3C[NH2+]C3)c(C3C[NH2+]C3)cc2c(=O...,3,assigned,0,|net_charge|=3 >= 3 (charge suspect)
349,pool_mol_350,CNc1cncc(N2CCC[C@H](N3CCC[C@H](N4CCCOC(C)C4)C3...,CNc1cncc(N2CCC[C@H]([NH+]3CCC[C@H](N4CCCOC(C)C...,1,assigned,1,1 undefined stereocentre(s); 1 phantom centre(...


## 6 · 落盘状态(仅 `DRY_RUN=False` 时 `n1.run` 已写盘)

In [8]:
if DRY_RUN:
    print("DRY_RUN=True:未写盘。核验第 4–5 步无误后,将顶部 DRY_RUN 改为 False 重跑本 notebook。")
else:
    assert OUT_CSV.exists() and OUT_MANIFEST.exists() and OUT_TRACE.exists(), "落盘不完整"
    tr = json.loads(OUT_TRACE.read_text(encoding="utf-8"))
    print("已写 :", OUT_CSV)
    print("输入 sha256 :", tr["input_sha256"])
    print("输出 sha256 :", tr["output_sha256"])
    print("manifest    :", OUT_MANIFEST)
    print("trace       :", OUT_TRACE)

已写 : /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/master_microstates.csv
输入 sha256 : 64c1d88ad3986ef4de9a775af9a7c0b24b965170bc3ec643469d3045f2196e78
输出 sha256 : b7abb39bf4dae099bb61a6f304e69fea8eb9755def8fd84176b3754953388170
manifest    : /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/master_microstates.manifest.json
trace       : /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/master_microstates.trace.json


## 7 · 节点状态板 & 下一步

- **产物**:`results/01_microstates/master_microstates.csv`(每分子一个权威微观态 + 净电荷 + 复核标记 + `qm_final_status`)
- **本步固定**:权威**电荷/微观态**(pH 7.4,Dimorphite,仅质子化)
- **待人工**:`needs_review==True` 的分子(多态歧义 / 未定义立体 / 互变异构)——在进入 Node 02 前定夺
- **下一步**:**Node 02** — 用确定的微观态生成 3D 主分子(master SDF),并立起 `CanonicalIsoSMILES(derived)==CanonicalIsoSMILES(master)` 不变量
- **待办(QM)**:`qm_final_status=="recompute"` 的分子,DFT 电子性质须按权威电荷重算
